In [0]:
create catalog if not exists Brazilian_E_Commerce;

In [0]:
create schema if not exists brazilian_e_commerce.Sql_practice;

In [0]:
create table if not exists brazilian_e_commerce.sql_practice.olist_orders_dataset 
as select * from read_files("/Volumes/brazilian_e_commerce/sql_practice/brazilian-ecommerce/olist_orders_dataset.csv")

In [0]:
describe brazilian_e_commerce.sql_practice.olist_orders_dataset;


## select / where

### --1.Retrieve all columns for orders in olist_orders_dataset where the order_status is 'delivered'.

In [0]:
select * from brazilian_e_commerce.sql_practice.olist_orders_dataset
where order_status = "delivered";

### 2.Find all customers in olist_customers_dataset who are located in the state of 'SP' (São Paulo).

In [0]:
create table if not exists brazilian_e_commerce.sql_practice.olist_customers_dataset 
as select * from read_files("/Volumes/brazilian_e_commerce/sql_practice/brazilian-ecommerce/olist_customers_dataset.csv");
describe brazilian_e_commerce.sql_practice.olist_customers_dataset;


In [0]:
select * from brazilian_e_commerce.sql_practice.olist_customers_dataset
where customer_state = "SP";

### 3.Find all order items in olist_order_items_dataset where the price is greater than 100 and freight_value is less than 20.

In [0]:
create table if not exists brazilian_e_commerce.sql_practice.olist_order_items_dataset 
as select * from read_files("/Volumes/brazilian_e_commerce/sql_practice/brazilian-ecommerce/olist_order_items_dataset.csv");
describe brazilian_e_commerce.sql_practice.olist_order_items_dataset;

In [0]:
select * from brazilian_e_commerce.sql_practice.olist_order_items_dataset
where price > 100 and freight_value < 20;

## Group by

### 4.Calculate the total number of orders placed per order_status in olist_orders_dataset.

In [0]:
select order_status,count(order_id) from brazilian_e_commerce.sql_practice.olist_orders_dataset
group by order_status;

### 5.Find the average price and maximum freight_value for each product_id in olist_order_items_dataset

In [0]:
select product_id,avg(price),max(freight_value) from brazilian_e_commerce.sql_practice.olist_order_items_dataset
group by product_id;

### 6.Count the number of unique customers (customer_unique_id) in each customer_state from olist_customers_dataset. 1

In [0]:
select customer_state,count(customer_unique_id) from brazilian_e_commerce.sql_practice.olist_customers_dataset
group by customer_state;

## HAVING
### 7.Find all product_ids in olist_order_items_dataset that have been ordered more than 50 times.

In [0]:
select product_id,count(order_id) as order_count from brazilian_e_commerce.sql_practice.olist_order_items_dataset
group by product_id
having count(order_id) > 50
order by order_count desc;

### 8.Identify customer_states in olist_customers_dataset that have strictly more than 5,000 customers.

In [0]:
select customer_state,count(customer_id) as total_customers from brazilian_e_commerce.sql_practice.olist_customers_dataset
group by customer_state
having count(customer_unique_id) > 5000;

### 9.List seller IDs (seller_id) from olist_order_items_dataset whose total sales revenue (sum of price) exceeds 10,000.

In [0]:
select seller_id,sum(price) as total_revenue from brazilian_e_commerce.sql_practice.olist_order_items_dataset
group by seller_id
having sum(price) > 10000
order by total_revenue desc;

## JOIN
### 10.Join olist_orders_dataset and olist_customers_dataset on customer_id to retrieve order_id, order_status, and customer_city

In [0]:
select ord.order_id,ord.order_status,cus.customer_city from brazilian_e_commerce.sql_practice.olist_customers_dataset cus join brazilian_e_commerce.sql_practice.olist_orders_dataset ord on cus.customer_id = ord.customer_id;

### 11.Join olist_order_items_dataset with olist_products_dataset on product_id to get order_id, product_id, and product_category_name.

In [0]:
create table if not exists brazilian_e_commerce.sql_practice.olist_products_dataset as 
select * from read_files("/Volumes/brazilian_e_commerce/sql_practice/brazilian-ecommerce/olist_products_dataset.csv");
desc brazilian_e_commerce.sql_practice.olist_products_dataset;

In [0]:
select orditm.order_id,prod.product_id,prod.product_category_name from brazilian_e_commerce.sql_practice.olist_order_items_dataset orditm join brazilian_e_commerce.sql_practice.olist_products_dataset prod on orditm.product_id = prod.product_id;

## JOIN + GROUP BY
### 12.Find the total number of orders placed by customers in each customer_city

In [0]:
select cus.customer_city,count(ord.order_id) from brazilian_e_commerce.sql_practice.olist_customers_dataset cus join brazilian_e_commerce.sql_practice.olist_orders_dataset ord on cus.customer_id = ord.customer_id group by cus.customer_city;

##JOIN + GROUP BY + HAVING
### 13.Get the top product categories by total sales amount (price) that have generated more than 50,000 in total revenue. Join olist_order_items_dataset and olist_products_dataset.

In [0]:
select prod.product_category_name,round(sum(orditm.price),2) total_sales_amount
from brazilian_e_commerce.sql_practice.olist_order_items_dataset orditm join brazilian_e_commerce.sql_practice.olist_products_dataset prod on orditm.product_id = prod.product_id 
group by prod.product_category_name
having sum(orditm.price) > 50000
order by total_sales_amount desc;

##JOIN + WHERE + GROUP BY
### 14.Calculate the total price of all 'delivered' orders for each seller (seller_id). Join olist_orders_dataset with olist_order_items_dataset.

In [0]:
select orditm.seller_id,sum(orditm.price) 
from brazilian_e_commerce.sql_practice.olist_order_items_dataset orditm join brazilian_e_commerce.sql_practice.olist_orders_dataset ord on orditm.order_id = ord.order_id 
where ord.order_status = 'delivered'
group by orditm.seller_id
order by sum(orditm.price) desc;

## JOIN + WHERE + GROUP BY + HAVING
### 15.Find all sellers (seller_id) who have delivered more than 100 orders, displaying their ID and total items sold.

In [0]:
select orditm.seller_id,count(orditm.order_id)  total_items_sold
from brazilian_e_commerce.sql_practice.olist_order_items_dataset orditm
join brazilian_e_commerce.sql_practice.olist_orders_dataset ord on orditm.order_id = ord.order_id 
where ord.order_status = 'delivered'
group by orditm.seller_id
having count(distinct orditm.order_id) > 100
order by total_items_sold desc;